# Week 06 Learning Notebook

This notebook mirrors the daily lesson plan and includes runnable demo code cells.

In [1]:
from datetime import date
import warnings
import numpy as np
import pandas as pd

try:
    import yfinance as yf
except Exception:
    yf = None

try:
    from pandas_datareader import data as pdr
except Exception:
    pdr = None

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.6f}'.format

def _synthetic_price_frame(tickers, start='2018-01-01', periods=900, seed=7):
    idx = pd.bdate_range(start, periods=periods)
    out = pd.DataFrame(index=idx)
    rng = np.random.default_rng(seed)
    for i, ticker in enumerate(tickers):
        drift = 0.0002 + 0.00005 * (i + 1)
        vol = 0.010 + 0.002 * i
        rets = rng.normal(drift, vol, size=len(idx))
        out[ticker] = 100.0 * np.exp(np.cumsum(rets))
    return out

def load_market_prices(tickers, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    tickers = list(tickers)
    if yf is None:
        warnings.warn('yfinance unavailable, using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

    try:
        raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
        if isinstance(raw.columns, pd.MultiIndex):
            if 'Close' in raw.columns.get_level_values(0):
                close = raw['Close']
            else:
                close = raw.xs('Close', axis=1, level=0, drop_level=True)
        else:
            close = raw.rename(columns={raw.columns[0]: tickers[0]})
        close = close.reindex(columns=tickers)
        close = close.dropna(how='all').ffill().dropna()
        if close.empty:
            raise ValueError('empty market data from Yahoo Finance')
        return close
    except Exception as exc:
        warnings.warn(f'Yahoo download failed ({exc}); using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

def load_fred_series(series_id, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    if pdr is None:
        warnings.warn('pandas_datareader unavailable, using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

    try:
        ser = pdr.DataReader(series_id, 'fred', start, end)[series_id]
        ser = ser.ffill().dropna()
        if ser.empty:
            raise ValueError('empty FRED series')
        return ser
    except Exception as exc:
        warnings.warn(f'FRED download failed ({exc}); using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

# Week 06 Day 01: Decision trees and feature interactions

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours (core + required extension); optional deep work extends to 10 hours.

## 6-10 Hour Daily Structure
- **Core Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Core Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Core Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Core Block 4 (60 min):** Python/pandas implementation and output verification.
- **Core Block 5 (30 min):** Practice questions, interview drill, and reflection.
- **Required Extension Block A (60 min):** Re-run the real trading example with one alternate ticker and one stress window.
- **Required Extension Block B (60 min):** Restart kernel and rerun all coding cells end-to-end, then add one extra validation test.
- **Optional Deep Work (0-4 hours):** Expand diagnostics, add edge-case tests, and improve interview-ready explanations.

## Why It Matters in Quant
Advance supervised modeling with non-linear learners and robust model diagnostics.

## Continuity and Handoff
- Previous checkpoint: Week 05 Day 07: Portfolio Project
- Previous lesson file: content/week-05/day-07.md
- Today's deliverable: Fit and visualize shallow vs deep trees with validation scores.
- Next handoff target: Week 06 Day 02: Random forests
- Next lesson file: content/week-06/day-02.md

## Theory Concepts

### Concept 1: Non-linearity capture
Non-linearity capture is a core part of 'Supervised ML II: trees, ensembles, and diagnostics'. Start with notation discipline: define labels, feature windows, and leakage boundaries before fitting any model. Then focus on causal feature design, leakage control, and robust model validation by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Tree depth and complexity
Tree depth and complexity is a core part of 'Supervised ML II: trees, ensembles, and diagnostics'. Start with notation discipline: define labels, feature windows, and leakage boundaries before fitting any model. Then focus on causal feature design, leakage control, and robust model validation by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Interpretability via splits
Interpretability via splits is a core part of 'Supervised ML II: trees, ensembles, and diagnostics'. Start with notation discipline: define labels, feature windows, and leakage boundaries before fitting any model. Then focus on causal feature design, leakage control, and robust model validation by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Forward Target
$$
y_t=\mathbb{1}[r_{t+1}>0]
$$
**Plain-English interpretation:** Label must stay forward-looking.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Use a walk-forward split and verify the metric does not leak future labels into feature construction.

### Formula 2: Logistic Probability
$$
p=\frac{1}{1+e^{-z}}
$$
**Plain-English interpretation:** Convert score to probability.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Use a walk-forward split and verify the metric does not leak future labels into feature construction.

### Formula 3: MSE
$$
\mathcal{L}_{MSE}=\frac{1}{n}\sum_i(y_i-\hat{y}_i)^2
$$
**Plain-English interpretation:** Baseline regression loss.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Use a walk-forward split and verify the metric does not leak future labels into feature construction.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $\hat{y}$ | Model prediction | class/probability | 0.73 |
| $\lambda$ | Regularization strength | non-negative scalar | 0.10 |
| $TP,FP,FN$ | Confusion matrix counts | integer count | TP=52 |

## Real Trading Example
- Instruments: SPY, XLK, XLF, XLE
- Macro overlay (FRED): DFF, VIXCLS
- Suggested window: 2018-01-01 to 2026-03-31
- Stress windows to inspect: 2020-11 to 2021-03, 2022-06 to 2022-11
- Scenario context: post-earnings dispersion with elevated feature drift
- Day objective: Train a decision tree and inspect split logic for feature intuition.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull listed FRED series and join strictly by release-aware timestamps (no look-ahead).
3. Compute today's formulas and compare calm vs stress-window behavior.
4. Translate outputs into one explicit trade action and one hard risk guardrail.
5. Validate that the decision is consistent with topic 'Decision trees and feature interactions'.

## Step-by-Step Solved Problems
### Solved Problem 1: Convert logit to probability
Given:
- Model score z = -0.400.
Solution:
1. $p=\frac{1}{1+e^{-z}}$.
2. p = 1/(1+exp(--0.400)) = 0.401312.
Final answer: Predicted probability = 40.13%.
Common trap: Using forward labels with backward-filled features, which silently introduces leakage.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Compute precision, recall, F1
Given:
- TP=50, FP=12, FN=18.
Solution:
1. $Precision=\frac{TP}{TP+FP}$.
2. Precision=0.8065.
3. $Recall=\frac{TP}{TP+FN}$.
4. Recall=0.7353.
5. $F1=\frac{2PR}{P+R}$.
6. F1=0.7692.
Final answer: Precision=80.65%, Recall=73.53%, F1=76.92%.
Common trap: Using forward labels with backward-filled features, which silently introduces leakage.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Compute ridge objective
Given:
- MSE = 0.0410, ||beta||_2^2 = 1.90, lambda = 0.06.
Solution:
1. $\mathcal{L}=\mathcal{L}_{MSE}+\lambda\|\beta\|_2^2$.
2. L = 0.0410 + 0.06*1.90 = 0.1550.
Final answer: Ridge objective = 0.1550.
Common trap: Using forward labels with backward-filled features, which silently introduces leakage.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Fit and visualize shallow vs deep trees with validation scores.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
X_train, X_valid, y_train, y_valid = temporal_split(features, target)
model = fit_logistic_regression(X_train, y_train, l2=0.1)
proba = model.predict_proba(X_valid)[:, 1]
report = classification_report_from_thresholds(y_valid, proba)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 06 Day 01): Explain Forward Target and define every symbol clearly for a post-earnings momentum regime where feature drift is elevated.
   - Model answer: "I use Forward Target as a decision bridge from market observations to position sizing. The formula is $y_t=\mathbb{1}[r_{t+1}>0]$. I define each symbol with units first, then compute one concrete value, and finally state what trade action changes because of the result in this regime."
2. Risk manager question: Using one real ticker from this lesson, what hard guardrail would you enforce before live deployment?
   - Model answer: "I would run the workflow on SPY and a stress-sensitive peer, then freeze new model entries when calibration error worsens for 3 consecutive windows. If the guardrail triggers, I switch to paper-trade monitoring and block new risk until diagnostics re-pass."
3. Data integrity question: Which checks must pass before you trust the output and place risk?
   - Model answer: "Before trading I verify strict train/validation timestamp split, no forward-filled target leakage, and feature freshness checks. If any check fails, I classify the run as non-tradable and log the incident."
4. Production question: Why does 'Decision trees and feature interactions' matter in live trading systems?
   - Model answer: "Decision trees and feature interactions matters because model decisions fail quickly without leakage controls, drift monitoring, and threshold governance. In production I need reproducible calculations, explicit control limits, and escalation rules that survive stress windows."
5. Decision question: If your key metric degrades for three consecutive sessions, what is your fallback plan?
   - Model answer: "I pause new signals, revert to baseline model, and relaunch only after calibration recovers. I only restore risk after rerun evidence confirms that assumptions are stable again."

Scoring rubric:
- 10/10: correct notation, one numeric example, explicit guardrail, data checks, and escalation path.
- 8/10: mostly correct notation plus a clear guardrail and fallback action.
- 6/10: partial correctness but vague controls or missing data validation.
- Below 6/10: formula recall without decision-quality risk controls.

### Interview Drill
- Prompt: "Walk me through Decision trees and feature interactions in a model-risk review after prediction drift and weaker precision."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision under constraints.
  3. Awareness of edge cases, costs, and failure modes.
  4. Clear escalation rule when guardrails are breached.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula, assumptions, and validation checks clearly.
  - Decision: explain one actionable rule, one risk guardrail, and one fallback action.

## Required Extension Track (2+ Hours)
- Re-run today's notebook from a fresh kernel so outputs are reproducible without hidden state.
- Add one additional risk guardrail and verify how it changes trade/no-trade decisions.
- Document one failure mode, one mitigation, and one escalation rule for production use.

Extension completion checks:
- [ ] Notebook restarted and all coding cells rerun successfully
- [ ] At least one extra validation/edge-case test added
- [ ] Risk guardrail and fallback action documented

## Reflection Question
How does tree depth influence regime sensitivity?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Full notebook rerun completed from clean kernel
- [ ] Reflection logged in progress tracker


## Week 06 Day 01 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [2]:
# ML demo: directional classifier from real SPY features
np.random.seed(601)
spy = load_market_prices(['SPY'], start='2014-01-01')['SPY']
ret = spy.pct_change()

feat = pd.DataFrame(index=spy.index)
feat['lag_1'] = ret.shift(1)
feat['lag_5'] = ret.rolling(5).mean().shift(1)
feat['vol_20'] = ret.rolling(20).std().shift(1)
feat['macro_rate'] = load_fred_series('DFF', start='2014-01-01').reindex(feat.index).ffill()
target = (ret.shift(-1) > 0).astype(int)
dataset = feat.join(target.rename('target')).dropna()

split = int(0.8 * len(dataset))
train = dataset.iloc[:split]
test = dataset.iloc[split:]

try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, roc_auc_score

    model = LogisticRegression(max_iter=500)
    model.fit(train.drop(columns=['target']), train['target'])
    proba = model.predict_proba(test.drop(columns=['target']))[:, 1]
    pred = (proba >= 0.5).astype(int)
    out = {
        'accuracy': float(accuracy_score(test['target'], pred)),
        'auc': float(roc_auc_score(test['target'], proba)),
    }
except Exception:
    # fallback classifier if sklearn is unavailable
    pred = (test['lag_1'] > 0).astype(int)
    out = {
        'accuracy': float((pred.values == test['target'].values).mean()),
        'auc': float('nan'),
    }

out


/var/folders/7l/31hylb_513bbgfbz8nnsgp840000gn/T/ipykernel_85036/2091198247.py:58: UserWarning: pandas_datareader unavailable, using synthetic macro data
  warnings.warn('pandas_datareader unavailable, using synthetic macro data')


{'accuracy': 0.5814332247557004, 'auc': 0.5240929056447482}

## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [3]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(9601)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Decision trees and feature interactions',
    'week': 6,
    'day': 1,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Decision trees and feature interactions',
 'week': 6,
 'day': 1,
 'observe_annual_return': 0.14923385976701686,
 'observe_annual_vol': 0.20501274117130058,
 'reason_sharpe_proxy': 0.581592437064142,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 06 Day 01 Quiz

Topic: **Decision trees and feature interactions**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Explain Logistic Probability (p = 1 / (1 + exp(-z))) and define every symbol clearly.
2. Risk question: What hard guardrail would you enforce before deployment?
3. Data question: Which validation checks must pass before trusting this output?
4. Production question: Why does this topic matter in live systems, not just in notebooks?
5. Escalation question: If the metric degrades for three sessions, what is your fallback action?

Hint for Q2: pause new model entries when calibration error degrades for three windows.
Hint for Q3: feature freshness and no target leakage
Scoring: full credit requires notation correctness, one numeric example, explicit guardrail, and fallback path.

In [4]:
# Quiz model answers (auto-generated)
z = -0.2700
metric_value = 1 / (1 + np.exp(-z))
metric_name = 'up_probability'

print('Interview Question 1 (model answer):')
print('  Formula focus: Logistic Probability')
print('  Formula text : p = 1 / (1 + exp(-z))')
print('  Scenario     : post-earnings drift regime')
print('  Numeric value:', round(float(metric_value), 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  freeze entries if calibration error worsens across three checks.')

print('\nInterview Question 3 (model answer):')
print('  I validate timestamps, missing data handling, and assumption consistency before taking risk.')

print('\nInterview Question 4 (model answer):')
print('  Topic:', 'Decision trees and feature interactions')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nInterview Question 5 (model answer):')
print('  Fallback action: revert to baseline model and block new entries.')

print('\nNumeric verification:')
print('  metric_name  =', metric_name)
print('  metric_value =', round(float(metric_value), 6))


Interview Question 1 (model answer):
  Formula focus: Logistic Probability
  Formula text : p = 1 / (1 + exp(-z))
  Scenario     : post-earnings drift regime
  Numeric value: 0.432907

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  freeze entries if calibration error worsens across three checks.

Interview Question 3 (model answer):
  I validate timestamps, missing data handling, and assumption consistency before taking risk.

Interview Question 4 (model answer):
  Topic: Decision trees and feature interactions
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Interview Question 5 (model answer):
  Fallback action: revert to baseline model and block new entries.

Numeric verification:
  metric_name  = up_probability
  metric_value = 0.432907


# Week 06 Day 02: Random forests

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours (core + required extension); optional deep work extends to 10 hours.

## 6-10 Hour Daily Structure
- **Core Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Core Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Core Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Core Block 4 (60 min):** Python/pandas implementation and output verification.
- **Core Block 5 (30 min):** Practice questions, interview drill, and reflection.
- **Required Extension Block A (60 min):** Re-run the real trading example with one alternate ticker and one stress window.
- **Required Extension Block B (60 min):** Restart kernel and rerun all coding cells end-to-end, then add one extra validation test.
- **Optional Deep Work (0-4 hours):** Expand diagnostics, add edge-case tests, and improve interview-ready explanations.

## Why It Matters in Quant
Advance supervised modeling with non-linear learners and robust model diagnostics.

## Continuity and Handoff
- Previous checkpoint: Week 06 Day 01: Decision trees and feature interactions
- Previous lesson file: content/week-06/day-01.md
- Today's deliverable: Train random forest and analyze feature importance shifts over time.
- Next handoff target: Week 06 Day 03: Gradient boosting intuition
- Next lesson file: content/week-06/day-03.md

## Theory Concepts

### Concept 1: Bagging and variance reduction
Bagging and variance reduction is a core part of 'Supervised ML II: trees, ensembles, and diagnostics'. Start with notation discipline: define labels, feature windows, and leakage boundaries before fitting any model. Then focus on causal feature design, leakage control, and robust model validation by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Out-of-bag intuition
Out-of-bag intuition is a core part of 'Supervised ML II: trees, ensembles, and diagnostics'. Start with notation discipline: define labels, feature windows, and leakage boundaries before fitting any model. Then focus on causal feature design, leakage control, and robust model validation by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Feature importance caveats
Feature importance caveats is a core part of 'Supervised ML II: trees, ensembles, and diagnostics'. Start with notation discipline: define labels, feature windows, and leakage boundaries before fitting any model. Then focus on causal feature design, leakage control, and robust model validation by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Logistic Probability
$$
p=\frac{1}{1+e^{-z}}
$$
**Plain-English interpretation:** Convert score to probability.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Use a walk-forward split and verify the metric does not leak future labels into feature construction.

### Formula 2: MSE
$$
\mathcal{L}_{MSE}=\frac{1}{n}\sum_i(y_i-\hat{y}_i)^2
$$
**Plain-English interpretation:** Baseline regression loss.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Use a walk-forward split and verify the metric does not leak future labels into feature construction.

### Formula 3: Cross-Entropy
$$
\mathcal{L}_{CE}=-\frac{1}{n}\sum_i[y_i\log p_i+(1-y_i)\log(1-p_i)]
$$
**Plain-English interpretation:** Classification objective.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Use a walk-forward split and verify the metric does not leak future labels into feature construction.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $\hat{y}$ | Model prediction | class/probability | 0.73 |
| $\lambda$ | Regularization strength | non-negative scalar | 0.10 |
| $TP,FP,FN$ | Confusion matrix counts | integer count | TP=52 |

## Real Trading Example
- Instruments: SPY, XLK, XLF, XLE
- Macro overlay (FRED): DFF, VIXCLS
- Suggested window: 2018-01-01 to 2026-03-31
- Stress windows to inspect: 2020-11 to 2021-03, 2022-06 to 2022-11
- Scenario context: post-earnings dispersion with elevated feature drift
- Day objective: Compare random forest stability across bootstrap samples.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull listed FRED series and join strictly by release-aware timestamps (no look-ahead).
3. Compute today's formulas and compare calm vs stress-window behavior.
4. Translate outputs into one explicit trade action and one hard risk guardrail.
5. Validate that the decision is consistent with topic 'Random forests'.

## Step-by-Step Solved Problems
### Solved Problem 1: Convert logit to probability
Given:
- Model score z = -0.100.
Solution:
1. $p=\frac{1}{1+e^{-z}}$.
2. p = 1/(1+exp(--0.100)) = 0.475021.
Final answer: Predicted probability = 47.50%.
Common trap: Using forward labels with backward-filled features, which silently introduces leakage.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Compute precision, recall, F1
Given:
- TP=50, FP=12, FN=18.
Solution:
1. $Precision=\frac{TP}{TP+FP}$.
2. Precision=0.8065.
3. $Recall=\frac{TP}{TP+FN}$.
4. Recall=0.7353.
5. $F1=\frac{2PR}{P+R}$.
6. F1=0.7692.
Final answer: Precision=80.65%, Recall=73.53%, F1=76.92%.
Common trap: Using forward labels with backward-filled features, which silently introduces leakage.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Compute ridge objective
Given:
- MSE = 0.0410, ||beta||_2^2 = 1.90, lambda = 0.06.
Solution:
1. $\mathcal{L}=\mathcal{L}_{MSE}+\lambda\|\beta\|_2^2$.
2. L = 0.0410 + 0.06*1.90 = 0.1550.
Final answer: Ridge objective = 0.1550.
Common trap: Using forward labels with backward-filled features, which silently introduces leakage.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Train random forest and analyze feature importance shifts over time.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
X_train, X_valid, y_train, y_valid = temporal_split(features, target)
model = fit_logistic_regression(X_train, y_train, l2=0.1)
proba = model.predict_proba(X_valid)[:, 1]
report = classification_report_from_thresholds(y_valid, proba)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 06 Day 02): Explain Logistic Probability and define every symbol clearly for a post-earnings momentum regime where feature drift is elevated.
   - Model answer: "I use Logistic Probability as a decision bridge from market observations to position sizing. The formula is $p=\frac{1}{1+e^{-z}}$. I define each symbol with units first, then compute one concrete value, and finally state what trade action changes because of the result in this regime."
2. Risk manager question: Using one real ticker from this lesson, what hard guardrail would you enforce before live deployment?
   - Model answer: "I would run the workflow on SPY and a stress-sensitive peer, then freeze new model entries when calibration error worsens for 3 consecutive windows. If the guardrail triggers, I switch to paper-trade monitoring and block new risk until diagnostics re-pass."
3. Data integrity question: Which checks must pass before you trust the output and place risk?
   - Model answer: "Before trading I verify strict train/validation timestamp split, no forward-filled target leakage, and feature freshness checks. If any check fails, I classify the run as non-tradable and log the incident."
4. Production question: Why does 'Random forests' matter in live trading systems?
   - Model answer: "Random forests matters because model decisions fail quickly without leakage controls, drift monitoring, and threshold governance. In production I need reproducible calculations, explicit control limits, and escalation rules that survive stress windows."
5. Decision question: If your key metric degrades for three consecutive sessions, what is your fallback plan?
   - Model answer: "I pause new signals, revert to baseline model, and relaunch only after calibration recovers. I only restore risk after rerun evidence confirms that assumptions are stable again."

Scoring rubric:
- 10/10: correct notation, one numeric example, explicit guardrail, data checks, and escalation path.
- 8/10: mostly correct notation plus a clear guardrail and fallback action.
- 6/10: partial correctness but vague controls or missing data validation.
- Below 6/10: formula recall without decision-quality risk controls.

### Interview Drill
- Prompt: "Walk me through Random forests in a model-risk review after prediction drift and weaker precision."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision under constraints.
  3. Awareness of edge cases, costs, and failure modes.
  4. Clear escalation rule when guardrails are breached.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula, assumptions, and validation checks clearly.
  - Decision: explain one actionable rule, one risk guardrail, and one fallback action.

## Required Extension Track (2+ Hours)
- Re-run today's notebook from a fresh kernel so outputs are reproducible without hidden state.
- Add one additional risk guardrail and verify how it changes trade/no-trade decisions.
- Document one failure mode, one mitigation, and one escalation rule for production use.

Extension completion checks:
- [ ] Notebook restarted and all coding cells rerun successfully
- [ ] At least one extra validation/edge-case test added
- [ ] Risk guardrail and fallback action documented

## Reflection Question
Why can feature importance be unstable in correlated predictors?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Full notebook rerun completed from clean kernel
- [ ] Reflection logged in progress tracker


## Week 06 Day 02 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [5]:
# ML demo: directional classifier from real SPY features
np.random.seed(602)
spy = load_market_prices(['SPY'], start='2014-01-01')['SPY']
ret = spy.pct_change()

feat = pd.DataFrame(index=spy.index)
feat['lag_1'] = ret.shift(1)
feat['lag_5'] = ret.rolling(5).mean().shift(1)
feat['vol_20'] = ret.rolling(20).std().shift(1)
feat['macro_rate'] = load_fred_series('DFF', start='2014-01-01').reindex(feat.index).ffill()
target = (ret.shift(-1) > 0).astype(int)
dataset = feat.join(target.rename('target')).dropna()

split = int(0.8 * len(dataset))
train = dataset.iloc[:split]
test = dataset.iloc[split:]

try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, roc_auc_score

    model = LogisticRegression(max_iter=500)
    model.fit(train.drop(columns=['target']), train['target'])
    proba = model.predict_proba(test.drop(columns=['target']))[:, 1]
    pred = (proba >= 0.5).astype(int)
    out = {
        'accuracy': float(accuracy_score(test['target'], pred)),
        'auc': float(roc_auc_score(test['target'], proba)),
    }
except Exception:
    # fallback classifier if sklearn is unavailable
    pred = (test['lag_1'] > 0).astype(int)
    out = {
        'accuracy': float((pred.values == test['target'].values).mean()),
        'auc': float('nan'),
    }

out


/var/folders/7l/31hylb_513bbgfbz8nnsgp840000gn/T/ipykernel_85036/2091198247.py:58: UserWarning: pandas_datareader unavailable, using synthetic macro data
  warnings.warn('pandas_datareader unavailable, using synthetic macro data')


{'accuracy': 0.5814332247557004, 'auc': 0.5240929056447482}

## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [6]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(9602)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Random forests',
    'week': 6,
    'day': 2,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Random forests',
 'week': 6,
 'day': 2,
 'observe_annual_return': 0.14923385976701686,
 'observe_annual_vol': 0.20501274117130058,
 'reason_sharpe_proxy': 0.581592437064142,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 06 Day 02 Quiz

Topic: **Random forests**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Explain Logistic Probability (p = 1 / (1 + exp(-z))) and define every symbol clearly.
2. Risk question: What hard guardrail would you enforce before deployment?
3. Data question: Which validation checks must pass before trusting this output?
4. Production question: Why does this topic matter in live systems, not just in notebooks?
5. Escalation question: If the metric degrades for three sessions, what is your fallback action?

Hint for Q2: pause new model entries when calibration error degrades for three windows.
Hint for Q3: feature freshness and no target leakage
Scoring: full credit requires notation correctness, one numeric example, explicit guardrail, and fallback path.

In [7]:
# Quiz model answers (auto-generated)
z = -0.1900
metric_value = 1 / (1 + np.exp(-z))
metric_name = 'up_probability'

print('Interview Question 1 (model answer):')
print('  Formula focus: Logistic Probability')
print('  Formula text : p = 1 / (1 + exp(-z))')
print('  Scenario     : post-earnings drift regime')
print('  Numeric value:', round(float(metric_value), 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  freeze entries if calibration error worsens across three checks.')

print('\nInterview Question 3 (model answer):')
print('  I validate timestamps, missing data handling, and assumption consistency before taking risk.')

print('\nInterview Question 4 (model answer):')
print('  Topic:', 'Random forests')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nInterview Question 5 (model answer):')
print('  Fallback action: revert to baseline model and block new entries.')

print('\nNumeric verification:')
print('  metric_name  =', metric_name)
print('  metric_value =', round(float(metric_value), 6))


Interview Question 1 (model answer):
  Formula focus: Logistic Probability
  Formula text : p = 1 / (1 + exp(-z))
  Scenario     : post-earnings drift regime
  Numeric value: 0.452642

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  freeze entries if calibration error worsens across three checks.

Interview Question 3 (model answer):
  I validate timestamps, missing data handling, and assumption consistency before taking risk.

Interview Question 4 (model answer):
  Topic: Random forests
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Interview Question 5 (model answer):
  Fallback action: revert to baseline model and block new entries.

Numeric verification:
  metric_name  = up_probability
  metric_value = 0.452642


# Week 06 Day 03: Gradient boosting intuition

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours (core + required extension); optional deep work extends to 10 hours.

## 6-10 Hour Daily Structure
- **Core Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Core Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Core Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Core Block 4 (60 min):** Python/pandas implementation and output verification.
- **Core Block 5 (30 min):** Practice questions, interview drill, and reflection.
- **Required Extension Block A (60 min):** Re-run the real trading example with one alternate ticker and one stress window.
- **Required Extension Block B (60 min):** Restart kernel and rerun all coding cells end-to-end, then add one extra validation test.
- **Optional Deep Work (0-4 hours):** Expand diagnostics, add edge-case tests, and improve interview-ready explanations.

## Why It Matters in Quant
Advance supervised modeling with non-linear learners and robust model diagnostics.

## Continuity and Handoff
- Previous checkpoint: Week 06 Day 02: Random forests
- Previous lesson file: content/week-06/day-02.md
- Today's deliverable: Run a small boosting hyperparameter grid and log results.
- Next handoff target: Week 06 Day 04: Feature engineering and selection
- Next lesson file: content/week-06/day-04.md

## Theory Concepts

### Concept 1: Sequential error correction
Sequential error correction is a core part of 'Supervised ML II: trees, ensembles, and diagnostics'. Start with notation discipline: define labels, feature windows, and leakage boundaries before fitting any model. Then focus on causal feature design, leakage control, and robust model validation by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Learning rate and depth tradeoff
Learning rate and depth tradeoff is a core part of 'Supervised ML II: trees, ensembles, and diagnostics'. Start with notation discipline: define labels, feature windows, and leakage boundaries before fitting any model. Then focus on causal feature design, leakage control, and robust model validation by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Overfitting controls
Overfitting controls is a core part of 'Supervised ML II: trees, ensembles, and diagnostics'. Start with notation discipline: define labels, feature windows, and leakage boundaries before fitting any model. Then focus on causal feature design, leakage control, and robust model validation by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: MSE
$$
\mathcal{L}_{MSE}=\frac{1}{n}\sum_i(y_i-\hat{y}_i)^2
$$
**Plain-English interpretation:** Baseline regression loss.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Use a walk-forward split and verify the metric does not leak future labels into feature construction.

### Formula 2: Cross-Entropy
$$
\mathcal{L}_{CE}=-\frac{1}{n}\sum_i[y_i\log p_i+(1-y_i)\log(1-p_i)]
$$
**Plain-English interpretation:** Classification objective.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Use a walk-forward split and verify the metric does not leak future labels into feature construction.

### Formula 3: Ridge Penalty
$$
\mathcal{L}=\mathcal{L}_{MSE}+\lambda\|\beta\|_2^2
$$
**Plain-English interpretation:** Regularized stability control.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Use a walk-forward split and verify the metric does not leak future labels into feature construction.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $\hat{y}$ | Model prediction | class/probability | 0.73 |
| $\lambda$ | Regularization strength | non-negative scalar | 0.10 |
| $TP,FP,FN$ | Confusion matrix counts | integer count | TP=52 |

## Real Trading Example
- Instruments: SPY, XLK, XLF, XLE
- Macro overlay (FRED): DFF, VIXCLS
- Suggested window: 2018-01-01 to 2026-03-31
- Stress windows to inspect: 2020-11 to 2021-03, 2022-06 to 2022-11
- Scenario context: post-earnings dispersion with elevated feature drift
- Day objective: Tune gradient boosting hyperparameters for a balanced precision-recall tradeoff.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull listed FRED series and join strictly by release-aware timestamps (no look-ahead).
3. Compute today's formulas and compare calm vs stress-window behavior.
4. Translate outputs into one explicit trade action and one hard risk guardrail.
5. Validate that the decision is consistent with topic 'Gradient boosting intuition'.

## Step-by-Step Solved Problems
### Solved Problem 1: Convert logit to probability
Given:
- Model score z = 0.200.
Solution:
1. $p=\frac{1}{1+e^{-z}}$.
2. p = 1/(1+exp(-0.200)) = 0.549834.
Final answer: Predicted probability = 54.98%.
Common trap: Using forward labels with backward-filled features, which silently introduces leakage.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Compute precision, recall, F1
Given:
- TP=50, FP=12, FN=18.
Solution:
1. $Precision=\frac{TP}{TP+FP}$.
2. Precision=0.8065.
3. $Recall=\frac{TP}{TP+FN}$.
4. Recall=0.7353.
5. $F1=\frac{2PR}{P+R}$.
6. F1=0.7692.
Final answer: Precision=80.65%, Recall=73.53%, F1=76.92%.
Common trap: Using forward labels with backward-filled features, which silently introduces leakage.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Compute ridge objective
Given:
- MSE = 0.0410, ||beta||_2^2 = 1.90, lambda = 0.06.
Solution:
1. $\mathcal{L}=\mathcal{L}_{MSE}+\lambda\|\beta\|_2^2$.
2. L = 0.0410 + 0.06*1.90 = 0.1550.
Final answer: Ridge objective = 0.1550.
Common trap: Using forward labels with backward-filled features, which silently introduces leakage.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Run a small boosting hyperparameter grid and log results.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
X_train, X_valid, y_train, y_valid = temporal_split(features, target)
model = fit_logistic_regression(X_train, y_train, l2=0.1)
proba = model.predict_proba(X_valid)[:, 1]
report = classification_report_from_thresholds(y_valid, proba)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 06 Day 03): Explain MSE and define every symbol clearly for a post-earnings momentum regime where feature drift is elevated.
   - Model answer: "I use MSE as a decision bridge from market observations to position sizing. The formula is $\mathcal{L}_{MSE}=\frac{1}{n}\sum_i(y_i-\hat{y}_i)^2$. I define each symbol with units first, then compute one concrete value, and finally state what trade action changes because of the result in this regime."
2. Risk manager question: Using one real ticker from this lesson, what hard guardrail would you enforce before live deployment?
   - Model answer: "I would run the workflow on SPY and a stress-sensitive peer, then freeze new model entries when calibration error worsens for 3 consecutive windows. If the guardrail triggers, I switch to paper-trade monitoring and block new risk until diagnostics re-pass."
3. Data integrity question: Which checks must pass before you trust the output and place risk?
   - Model answer: "Before trading I verify strict train/validation timestamp split, no forward-filled target leakage, and feature freshness checks. If any check fails, I classify the run as non-tradable and log the incident."
4. Production question: Why does 'Gradient boosting intuition' matter in live trading systems?
   - Model answer: "Gradient boosting intuition matters because model decisions fail quickly without leakage controls, drift monitoring, and threshold governance. In production I need reproducible calculations, explicit control limits, and escalation rules that survive stress windows."
5. Decision question: If your key metric degrades for three consecutive sessions, what is your fallback plan?
   - Model answer: "I pause new signals, revert to baseline model, and relaunch only after calibration recovers. I only restore risk after rerun evidence confirms that assumptions are stable again."

Scoring rubric:
- 10/10: correct notation, one numeric example, explicit guardrail, data checks, and escalation path.
- 8/10: mostly correct notation plus a clear guardrail and fallback action.
- 6/10: partial correctness but vague controls or missing data validation.
- Below 6/10: formula recall without decision-quality risk controls.

### Interview Drill
- Prompt: "Walk me through Gradient boosting intuition in a model-risk review after prediction drift and weaker precision."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision under constraints.
  3. Awareness of edge cases, costs, and failure modes.
  4. Clear escalation rule when guardrails are breached.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula, assumptions, and validation checks clearly.
  - Decision: explain one actionable rule, one risk guardrail, and one fallback action.

## Required Extension Track (2+ Hours)
- Re-run today's notebook from a fresh kernel so outputs are reproducible without hidden state.
- Add one additional risk guardrail and verify how it changes trade/no-trade decisions.
- Document one failure mode, one mitigation, and one escalation rule for production use.

Extension completion checks:
- [ ] Notebook restarted and all coding cells rerun successfully
- [ ] At least one extra validation/edge-case test added
- [ ] Risk guardrail and fallback action documented

## Reflection Question
How do boosting gains change under noisy labels?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Full notebook rerun completed from clean kernel
- [ ] Reflection logged in progress tracker


## Week 06 Day 03 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [8]:
# ML demo: directional classifier from real SPY features
np.random.seed(603)
spy = load_market_prices(['SPY'], start='2014-01-01')['SPY']
ret = spy.pct_change()

feat = pd.DataFrame(index=spy.index)
feat['lag_1'] = ret.shift(1)
feat['lag_5'] = ret.rolling(5).mean().shift(1)
feat['vol_20'] = ret.rolling(20).std().shift(1)
feat['macro_rate'] = load_fred_series('DFF', start='2014-01-01').reindex(feat.index).ffill()
target = (ret.shift(-1) > 0).astype(int)
dataset = feat.join(target.rename('target')).dropna()

split = int(0.8 * len(dataset))
train = dataset.iloc[:split]
test = dataset.iloc[split:]

try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, roc_auc_score

    model = LogisticRegression(max_iter=500)
    model.fit(train.drop(columns=['target']), train['target'])
    proba = model.predict_proba(test.drop(columns=['target']))[:, 1]
    pred = (proba >= 0.5).astype(int)
    out = {
        'accuracy': float(accuracy_score(test['target'], pred)),
        'auc': float(roc_auc_score(test['target'], proba)),
    }
except Exception:
    # fallback classifier if sklearn is unavailable
    pred = (test['lag_1'] > 0).astype(int)
    out = {
        'accuracy': float((pred.values == test['target'].values).mean()),
        'auc': float('nan'),
    }

out


/var/folders/7l/31hylb_513bbgfbz8nnsgp840000gn/T/ipykernel_85036/2091198247.py:58: UserWarning: pandas_datareader unavailable, using synthetic macro data
  warnings.warn('pandas_datareader unavailable, using synthetic macro data')


{'accuracy': 0.5814332247557004, 'auc': 0.5240929056447482}

## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [9]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(9603)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Gradient boosting intuition',
    'week': 6,
    'day': 3,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Gradient boosting intuition',
 'week': 6,
 'day': 3,
 'observe_annual_return': 0.14923385976701686,
 'observe_annual_vol': 0.20501274117130058,
 'reason_sharpe_proxy': 0.581592437064142,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 06 Day 03 Quiz

Topic: **Gradient boosting intuition**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Explain Logistic Probability (p = 1 / (1 + exp(-z))) and define every symbol clearly.
2. Risk question: What hard guardrail would you enforce before deployment?
3. Data question: Which validation checks must pass before trusting this output?
4. Production question: Why does this topic matter in live systems, not just in notebooks?
5. Escalation question: If the metric degrades for three sessions, what is your fallback action?

Hint for Q2: pause new model entries when calibration error degrades for three windows.
Hint for Q3: feature freshness and no target leakage
Scoring: full credit requires notation correctness, one numeric example, explicit guardrail, and fallback path.

In [10]:
# Quiz model answers (auto-generated)
z = -0.1100
metric_value = 1 / (1 + np.exp(-z))
metric_name = 'up_probability'

print('Interview Question 1 (model answer):')
print('  Formula focus: Logistic Probability')
print('  Formula text : p = 1 / (1 + exp(-z))')
print('  Scenario     : post-earnings drift regime')
print('  Numeric value:', round(float(metric_value), 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  freeze entries if calibration error worsens across three checks.')

print('\nInterview Question 3 (model answer):')
print('  I validate timestamps, missing data handling, and assumption consistency before taking risk.')

print('\nInterview Question 4 (model answer):')
print('  Topic:', 'Gradient boosting intuition')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nInterview Question 5 (model answer):')
print('  Fallback action: revert to baseline model and block new entries.')

print('\nNumeric verification:')
print('  metric_name  =', metric_name)
print('  metric_value =', round(float(metric_value), 6))


Interview Question 1 (model answer):
  Formula focus: Logistic Probability
  Formula text : p = 1 / (1 + exp(-z))
  Scenario     : post-earnings drift regime
  Numeric value: 0.472528

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  freeze entries if calibration error worsens across three checks.

Interview Question 3 (model answer):
  I validate timestamps, missing data handling, and assumption consistency before taking risk.

Interview Question 4 (model answer):
  Topic: Gradient boosting intuition
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Interview Question 5 (model answer):
  Fallback action: revert to baseline model and block new entries.

Numeric verification:
  metric_name  = up_probability
  metric_value = 0.472528


# Week 06 Day 04: Feature engineering and selection

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours (core + required extension); optional deep work extends to 10 hours.

## 6-10 Hour Daily Structure
- **Core Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Core Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Core Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Core Block 4 (60 min):** Python/pandas implementation and output verification.
- **Core Block 5 (30 min):** Practice questions, interview drill, and reflection.
- **Required Extension Block A (60 min):** Re-run the real trading example with one alternate ticker and one stress window.
- **Required Extension Block B (60 min):** Restart kernel and rerun all coding cells end-to-end, then add one extra validation test.
- **Optional Deep Work (0-4 hours):** Expand diagnostics, add edge-case tests, and improve interview-ready explanations.

## Why It Matters in Quant
Advance supervised modeling with non-linear learners and robust model diagnostics.

## Continuity and Handoff
- Previous checkpoint: Week 06 Day 03: Gradient boosting intuition
- Previous lesson file: content/week-06/day-03.md
- Today's deliverable: Create feature ablation experiment table.
- Next handoff target: Week 06 Day 05: Model diagnostics and monitoring
- Next lesson file: content/week-06/day-05.md

## Theory Concepts

### Concept 1: Lagged and rolling features
Lagged and rolling features is a core part of 'Supervised ML II: trees, ensembles, and diagnostics'. Start with notation discipline: define labels, feature windows, and leakage boundaries before fitting any model. Then focus on causal feature design, leakage control, and robust model validation by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Interaction terms
Interaction terms is a core part of 'Supervised ML II: trees, ensembles, and diagnostics'. Start with notation discipline: define labels, feature windows, and leakage boundaries before fitting any model. Then focus on causal feature design, leakage control, and robust model validation by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Feature ablation testing
Feature ablation testing is a core part of 'Supervised ML II: trees, ensembles, and diagnostics'. Start with notation discipline: define labels, feature windows, and leakage boundaries before fitting any model. Then focus on causal feature design, leakage control, and robust model validation by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Cross-Entropy
$$
\mathcal{L}_{CE}=-\frac{1}{n}\sum_i[y_i\log p_i+(1-y_i)\log(1-p_i)]
$$
**Plain-English interpretation:** Classification objective.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Use a walk-forward split and verify the metric does not leak future labels into feature construction.

### Formula 2: Ridge Penalty
$$
\mathcal{L}=\mathcal{L}_{MSE}+\lambda\|\beta\|_2^2
$$
**Plain-English interpretation:** Regularized stability control.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Use a walk-forward split and verify the metric does not leak future labels into feature construction.

### Formula 3: Forward Target
$$
y_t=\mathbb{1}[r_{t+1}>0]
$$
**Plain-English interpretation:** Label must stay forward-looking.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Use a walk-forward split and verify the metric does not leak future labels into feature construction.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $\hat{y}$ | Model prediction | class/probability | 0.73 |
| $\lambda$ | Regularization strength | non-negative scalar | 0.10 |
| $TP,FP,FN$ | Confusion matrix counts | integer count | TP=52 |

## Real Trading Example
- Instruments: SPY, XLK, XLF, XLE
- Macro overlay (FRED): DFF, VIXCLS
- Suggested window: 2018-01-01 to 2026-03-31
- Stress windows to inspect: 2020-11 to 2021-03, 2022-06 to 2022-11
- Scenario context: post-earnings dispersion with elevated feature drift
- Day objective: Build and compare engineered feature sets for predictive lift.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull listed FRED series and join strictly by release-aware timestamps (no look-ahead).
3. Compute today's formulas and compare calm vs stress-window behavior.
4. Translate outputs into one explicit trade action and one hard risk guardrail.
5. Validate that the decision is consistent with topic 'Feature engineering and selection'.

## Step-by-Step Solved Problems
### Solved Problem 1: Convert logit to probability
Given:
- Model score z = 0.500.
Solution:
1. $p=\frac{1}{1+e^{-z}}$.
2. p = 1/(1+exp(-0.500)) = 0.622459.
Final answer: Predicted probability = 62.25%.
Common trap: Using forward labels with backward-filled features, which silently introduces leakage.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Compute precision, recall, F1
Given:
- TP=50, FP=12, FN=18.
Solution:
1. $Precision=\frac{TP}{TP+FP}$.
2. Precision=0.8065.
3. $Recall=\frac{TP}{TP+FN}$.
4. Recall=0.7353.
5. $F1=\frac{2PR}{P+R}$.
6. F1=0.7692.
Final answer: Precision=80.65%, Recall=73.53%, F1=76.92%.
Common trap: Using forward labels with backward-filled features, which silently introduces leakage.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Compute ridge objective
Given:
- MSE = 0.0410, ||beta||_2^2 = 1.90, lambda = 0.06.
Solution:
1. $\mathcal{L}=\mathcal{L}_{MSE}+\lambda\|\beta\|_2^2$.
2. L = 0.0410 + 0.06*1.90 = 0.1550.
Final answer: Ridge objective = 0.1550.
Common trap: Using forward labels with backward-filled features, which silently introduces leakage.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Create feature ablation experiment table.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
X_train, X_valid, y_train, y_valid = temporal_split(features, target)
model = fit_logistic_regression(X_train, y_train, l2=0.1)
proba = model.predict_proba(X_valid)[:, 1]
report = classification_report_from_thresholds(y_valid, proba)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 06 Day 04): Explain Cross-Entropy and define every symbol clearly for a post-earnings momentum regime where feature drift is elevated.
   - Model answer: "I use Cross-Entropy as a decision bridge from market observations to position sizing. The formula is $\mathcal{L}_{CE}=-\frac{1}{n}\sum_i[y_i\log p_i+(1-y_i)\log(1-p_i)]$. I define each symbol with units first, then compute one concrete value, and finally state what trade action changes because of the result in this regime."
2. Risk manager question: Using one real ticker from this lesson, what hard guardrail would you enforce before live deployment?
   - Model answer: "I would run the workflow on SPY and a stress-sensitive peer, then freeze new model entries when calibration error worsens for 3 consecutive windows. If the guardrail triggers, I switch to paper-trade monitoring and block new risk until diagnostics re-pass."
3. Data integrity question: Which checks must pass before you trust the output and place risk?
   - Model answer: "Before trading I verify strict train/validation timestamp split, no forward-filled target leakage, and feature freshness checks. If any check fails, I classify the run as non-tradable and log the incident."
4. Production question: Why does 'Feature engineering and selection' matter in live trading systems?
   - Model answer: "Feature engineering and selection matters because model decisions fail quickly without leakage controls, drift monitoring, and threshold governance. In production I need reproducible calculations, explicit control limits, and escalation rules that survive stress windows."
5. Decision question: If your key metric degrades for three consecutive sessions, what is your fallback plan?
   - Model answer: "I pause new signals, revert to baseline model, and relaunch only after calibration recovers. I only restore risk after rerun evidence confirms that assumptions are stable again."

Scoring rubric:
- 10/10: correct notation, one numeric example, explicit guardrail, data checks, and escalation path.
- 8/10: mostly correct notation plus a clear guardrail and fallback action.
- 6/10: partial correctness but vague controls or missing data validation.
- Below 6/10: formula recall without decision-quality risk controls.

### Interview Drill
- Prompt: "Walk me through Feature engineering and selection in a model-risk review after prediction drift and weaker precision."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision under constraints.
  3. Awareness of edge cases, costs, and failure modes.
  4. Clear escalation rule when guardrails are breached.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula, assumptions, and validation checks clearly.
  - Decision: explain one actionable rule, one risk guardrail, and one fallback action.

## Required Extension Track (2+ Hours)
- Re-run today's notebook from a fresh kernel so outputs are reproducible without hidden state.
- Add one additional risk guardrail and verify how it changes trade/no-trade decisions.
- Document one failure mode, one mitigation, and one escalation rule for production use.

Extension completion checks:
- [ ] Notebook restarted and all coding cells rerun successfully
- [ ] At least one extra validation/edge-case test added
- [ ] Risk guardrail and fallback action documented

## Reflection Question
Which engineered features seem most fragile across windows?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Full notebook rerun completed from clean kernel
- [ ] Reflection logged in progress tracker


## Week 06 Day 04 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [11]:
# ML demo: directional classifier from real SPY features
np.random.seed(604)
spy = load_market_prices(['SPY'], start='2014-01-01')['SPY']
ret = spy.pct_change()

feat = pd.DataFrame(index=spy.index)
feat['lag_1'] = ret.shift(1)
feat['lag_5'] = ret.rolling(5).mean().shift(1)
feat['vol_20'] = ret.rolling(20).std().shift(1)
feat['macro_rate'] = load_fred_series('DFF', start='2014-01-01').reindex(feat.index).ffill()
target = (ret.shift(-1) > 0).astype(int)
dataset = feat.join(target.rename('target')).dropna()

split = int(0.8 * len(dataset))
train = dataset.iloc[:split]
test = dataset.iloc[split:]

try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, roc_auc_score

    model = LogisticRegression(max_iter=500)
    model.fit(train.drop(columns=['target']), train['target'])
    proba = model.predict_proba(test.drop(columns=['target']))[:, 1]
    pred = (proba >= 0.5).astype(int)
    out = {
        'accuracy': float(accuracy_score(test['target'], pred)),
        'auc': float(roc_auc_score(test['target'], proba)),
    }
except Exception:
    # fallback classifier if sklearn is unavailable
    pred = (test['lag_1'] > 0).astype(int)
    out = {
        'accuracy': float((pred.values == test['target'].values).mean()),
        'auc': float('nan'),
    }

out


/var/folders/7l/31hylb_513bbgfbz8nnsgp840000gn/T/ipykernel_85036/2091198247.py:58: UserWarning: pandas_datareader unavailable, using synthetic macro data
  warnings.warn('pandas_datareader unavailable, using synthetic macro data')


{'accuracy': 0.5814332247557004, 'auc': 0.5240929056447482}

## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [12]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(9604)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Feature engineering and selection',
    'week': 6,
    'day': 4,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Feature engineering and selection',
 'week': 6,
 'day': 4,
 'observe_annual_return': 0.14923385976701686,
 'observe_annual_vol': 0.20501274117130058,
 'reason_sharpe_proxy': 0.581592437064142,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 06 Day 04 Quiz

Topic: **Feature engineering and selection**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Explain Logistic Probability (p = 1 / (1 + exp(-z))) and define every symbol clearly.
2. Risk question: What hard guardrail would you enforce before deployment?
3. Data question: Which validation checks must pass before trusting this output?
4. Production question: Why does this topic matter in live systems, not just in notebooks?
5. Escalation question: If the metric degrades for three sessions, what is your fallback action?

Hint for Q2: pause new model entries when calibration error degrades for three windows.
Hint for Q3: feature freshness and no target leakage
Scoring: full credit requires notation correctness, one numeric example, explicit guardrail, and fallback path.

In [13]:
# Quiz model answers (auto-generated)
z = -0.0300
metric_value = 1 / (1 + np.exp(-z))
metric_name = 'up_probability'

print('Interview Question 1 (model answer):')
print('  Formula focus: Logistic Probability')
print('  Formula text : p = 1 / (1 + exp(-z))')
print('  Scenario     : post-earnings drift regime')
print('  Numeric value:', round(float(metric_value), 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  freeze entries if calibration error worsens across three checks.')

print('\nInterview Question 3 (model answer):')
print('  I validate timestamps, missing data handling, and assumption consistency before taking risk.')

print('\nInterview Question 4 (model answer):')
print('  Topic:', 'Feature engineering and selection')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nInterview Question 5 (model answer):')
print('  Fallback action: revert to baseline model and block new entries.')

print('\nNumeric verification:')
print('  metric_name  =', metric_name)
print('  metric_value =', round(float(metric_value), 6))


Interview Question 1 (model answer):
  Formula focus: Logistic Probability
  Formula text : p = 1 / (1 + exp(-z))
  Scenario     : post-earnings drift regime
  Numeric value: 0.492501

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  freeze entries if calibration error worsens across three checks.

Interview Question 3 (model answer):
  I validate timestamps, missing data handling, and assumption consistency before taking risk.

Interview Question 4 (model answer):
  Topic: Feature engineering and selection
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Interview Question 5 (model answer):
  Fallback action: revert to baseline model and block new entries.

Numeric verification:
  metric_name  = up_probability
  metric_value = 0.492501


# Week 06 Day 05: Model diagnostics and monitoring

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours (core + required extension); optional deep work extends to 10 hours.

## 6-10 Hour Daily Structure
- **Core Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Core Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Core Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Core Block 4 (60 min):** Python/pandas implementation and output verification.
- **Core Block 5 (30 min):** Practice questions, interview drill, and reflection.
- **Required Extension Block A (60 min):** Re-run the real trading example with one alternate ticker and one stress window.
- **Required Extension Block B (60 min):** Restart kernel and rerun all coding cells end-to-end, then add one extra validation test.
- **Optional Deep Work (0-4 hours):** Expand diagnostics, add edge-case tests, and improve interview-ready explanations.

## Why It Matters in Quant
Advance supervised modeling with non-linear learners and robust model diagnostics.

## Continuity and Handoff
- Previous checkpoint: Week 06 Day 04: Feature engineering and selection
- Previous lesson file: content/week-06/day-04.md
- Today's deliverable: Implement drift diagnostics on feature distributions.
- Next handoff target: Week 06 Day 06: Revision Sprint
- Next lesson file: content/week-06/day-06.md

## Theory Concepts

### Concept 1: Calibration and reliability
Calibration and reliability is a core part of 'Supervised ML II: trees, ensembles, and diagnostics'. Start with notation discipline: define labels, feature windows, and leakage boundaries before fitting any model. Then focus on causal feature design, leakage control, and robust model validation by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Drift detection basics
Drift detection basics is a core part of 'Supervised ML II: trees, ensembles, and diagnostics'. Start with notation discipline: define labels, feature windows, and leakage boundaries before fitting any model. Then focus on causal feature design, leakage control, and robust model validation by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Error slicing by regime
Error slicing by regime is a core part of 'Supervised ML II: trees, ensembles, and diagnostics'. Start with notation discipline: define labels, feature windows, and leakage boundaries before fitting any model. Then focus on causal feature design, leakage control, and robust model validation by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Ridge Penalty
$$
\mathcal{L}=\mathcal{L}_{MSE}+\lambda\|\beta\|_2^2
$$
**Plain-English interpretation:** Regularized stability control.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Use a walk-forward split and verify the metric does not leak future labels into feature construction.

### Formula 2: Forward Target
$$
y_t=\mathbb{1}[r_{t+1}>0]
$$
**Plain-English interpretation:** Label must stay forward-looking.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Use a walk-forward split and verify the metric does not leak future labels into feature construction.

### Formula 3: Logistic Probability
$$
p=\frac{1}{1+e^{-z}}
$$
**Plain-English interpretation:** Convert score to probability.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Use a walk-forward split and verify the metric does not leak future labels into feature construction.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $\hat{y}$ | Model prediction | class/probability | 0.73 |
| $\lambda$ | Regularization strength | non-negative scalar | 0.10 |
| $TP,FP,FN$ | Confusion matrix counts | integer count | TP=52 |

## Real Trading Example
- Instruments: SPY, XLK, XLF, XLE
- Macro overlay (FRED): DFF, VIXCLS
- Suggested window: 2018-01-01 to 2026-03-31
- Stress windows to inspect: 2020-11 to 2021-03, 2022-06 to 2022-11
- Scenario context: post-earnings dispersion with elevated feature drift
- Day objective: Analyze model errors during high-volatility periods.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull listed FRED series and join strictly by release-aware timestamps (no look-ahead).
3. Compute today's formulas and compare calm vs stress-window behavior.
4. Translate outputs into one explicit trade action and one hard risk guardrail.
5. Validate that the decision is consistent with topic 'Model diagnostics and monitoring'.

## Step-by-Step Solved Problems
### Solved Problem 1: Convert logit to probability
Given:
- Model score z = 0.800.
Solution:
1. $p=\frac{1}{1+e^{-z}}$.
2. p = 1/(1+exp(-0.800)) = 0.689974.
Final answer: Predicted probability = 69.00%.
Common trap: Using forward labels with backward-filled features, which silently introduces leakage.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Compute precision, recall, F1
Given:
- TP=50, FP=12, FN=18.
Solution:
1. $Precision=\frac{TP}{TP+FP}$.
2. Precision=0.8065.
3. $Recall=\frac{TP}{TP+FN}$.
4. Recall=0.7353.
5. $F1=\frac{2PR}{P+R}$.
6. F1=0.7692.
Final answer: Precision=80.65%, Recall=73.53%, F1=76.92%.
Common trap: Using forward labels with backward-filled features, which silently introduces leakage.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Compute ridge objective
Given:
- MSE = 0.0410, ||beta||_2^2 = 1.90, lambda = 0.06.
Solution:
1. $\mathcal{L}=\mathcal{L}_{MSE}+\lambda\|\beta\|_2^2$.
2. L = 0.0410 + 0.06*1.90 = 0.1550.
Final answer: Ridge objective = 0.1550.
Common trap: Using forward labels with backward-filled features, which silently introduces leakage.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Implement drift diagnostics on feature distributions.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
X_train, X_valid, y_train, y_valid = temporal_split(features, target)
model = fit_logistic_regression(X_train, y_train, l2=0.1)
proba = model.predict_proba(X_valid)[:, 1]
report = classification_report_from_thresholds(y_valid, proba)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 06 Day 05): Explain Ridge Penalty and define every symbol clearly for a post-earnings momentum regime where feature drift is elevated.
   - Model answer: "I use Ridge Penalty as a decision bridge from market observations to position sizing. The formula is $\mathcal{L}=\mathcal{L}_{MSE}+\lambda\|\beta\|_2^2$. I define each symbol with units first, then compute one concrete value, and finally state what trade action changes because of the result in this regime."
2. Risk manager question: Using one real ticker from this lesson, what hard guardrail would you enforce before live deployment?
   - Model answer: "I would run the workflow on SPY and a stress-sensitive peer, then freeze new model entries when calibration error worsens for 3 consecutive windows. If the guardrail triggers, I switch to paper-trade monitoring and block new risk until diagnostics re-pass."
3. Data integrity question: Which checks must pass before you trust the output and place risk?
   - Model answer: "Before trading I verify strict train/validation timestamp split, no forward-filled target leakage, and feature freshness checks. If any check fails, I classify the run as non-tradable and log the incident."
4. Production question: Why does 'Model diagnostics and monitoring' matter in live trading systems?
   - Model answer: "Model diagnostics and monitoring matters because model decisions fail quickly without leakage controls, drift monitoring, and threshold governance. In production I need reproducible calculations, explicit control limits, and escalation rules that survive stress windows."
5. Decision question: If your key metric degrades for three consecutive sessions, what is your fallback plan?
   - Model answer: "I pause new signals, revert to baseline model, and relaunch only after calibration recovers. I only restore risk after rerun evidence confirms that assumptions are stable again."

Scoring rubric:
- 10/10: correct notation, one numeric example, explicit guardrail, data checks, and escalation path.
- 8/10: mostly correct notation plus a clear guardrail and fallback action.
- 6/10: partial correctness but vague controls or missing data validation.
- Below 6/10: formula recall without decision-quality risk controls.

### Interview Drill
- Prompt: "Walk me through Model diagnostics and monitoring in a model-risk review after prediction drift and weaker precision."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision under constraints.
  3. Awareness of edge cases, costs, and failure modes.
  4. Clear escalation rule when guardrails are breached.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula, assumptions, and validation checks clearly.
  - Decision: explain one actionable rule, one risk guardrail, and one fallback action.

## Required Extension Track (2+ Hours)
- Re-run today's notebook from a fresh kernel so outputs are reproducible without hidden state.
- Add one additional risk guardrail and verify how it changes trade/no-trade decisions.
- Document one failure mode, one mitigation, and one escalation rule for production use.

Extension completion checks:
- [ ] Notebook restarted and all coding cells rerun successfully
- [ ] At least one extra validation/edge-case test added
- [ ] Risk guardrail and fallback action documented

## Reflection Question
What monitoring trigger would force model retraining?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Full notebook rerun completed from clean kernel
- [ ] Reflection logged in progress tracker


## Week 06 Day 05 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [14]:
# ML demo: directional classifier from real SPY features
np.random.seed(605)
spy = load_market_prices(['SPY'], start='2014-01-01')['SPY']
ret = spy.pct_change()

feat = pd.DataFrame(index=spy.index)
feat['lag_1'] = ret.shift(1)
feat['lag_5'] = ret.rolling(5).mean().shift(1)
feat['vol_20'] = ret.rolling(20).std().shift(1)
feat['macro_rate'] = load_fred_series('DFF', start='2014-01-01').reindex(feat.index).ffill()
target = (ret.shift(-1) > 0).astype(int)
dataset = feat.join(target.rename('target')).dropna()

split = int(0.8 * len(dataset))
train = dataset.iloc[:split]
test = dataset.iloc[split:]

try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, roc_auc_score

    model = LogisticRegression(max_iter=500)
    model.fit(train.drop(columns=['target']), train['target'])
    proba = model.predict_proba(test.drop(columns=['target']))[:, 1]
    pred = (proba >= 0.5).astype(int)
    out = {
        'accuracy': float(accuracy_score(test['target'], pred)),
        'auc': float(roc_auc_score(test['target'], proba)),
    }
except Exception:
    # fallback classifier if sklearn is unavailable
    pred = (test['lag_1'] > 0).astype(int)
    out = {
        'accuracy': float((pred.values == test['target'].values).mean()),
        'auc': float('nan'),
    }

out


/var/folders/7l/31hylb_513bbgfbz8nnsgp840000gn/T/ipykernel_85036/2091198247.py:58: UserWarning: pandas_datareader unavailable, using synthetic macro data
  warnings.warn('pandas_datareader unavailable, using synthetic macro data')


{'accuracy': 0.5814332247557004, 'auc': 0.5240929056447482}

## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [15]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(9605)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Model diagnostics and monitoring',
    'week': 6,
    'day': 5,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Model diagnostics and monitoring',
 'week': 6,
 'day': 5,
 'observe_annual_return': 0.14923385976701686,
 'observe_annual_vol': 0.20501274117130058,
 'reason_sharpe_proxy': 0.581592437064142,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 06 Day 05 Quiz

Topic: **Model diagnostics and monitoring**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Explain Logistic Probability (p = 1 / (1 + exp(-z))) and define every symbol clearly.
2. Risk question: What hard guardrail would you enforce before deployment?
3. Data question: Which validation checks must pass before trusting this output?
4. Production question: Why does this topic matter in live systems, not just in notebooks?
5. Escalation question: If the metric degrades for three sessions, what is your fallback action?

Hint for Q2: pause new model entries when calibration error degrades for three windows.
Hint for Q3: feature freshness and no target leakage
Scoring: full credit requires notation correctness, one numeric example, explicit guardrail, and fallback path.

In [16]:
# Quiz model answers (auto-generated)
z = 0.0500
metric_value = 1 / (1 + np.exp(-z))
metric_name = 'up_probability'

print('Interview Question 1 (model answer):')
print('  Formula focus: Logistic Probability')
print('  Formula text : p = 1 / (1 + exp(-z))')
print('  Scenario     : post-earnings drift regime')
print('  Numeric value:', round(float(metric_value), 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  freeze entries if calibration error worsens across three checks.')

print('\nInterview Question 3 (model answer):')
print('  I validate timestamps, missing data handling, and assumption consistency before taking risk.')

print('\nInterview Question 4 (model answer):')
print('  Topic:', 'Model diagnostics and monitoring')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nInterview Question 5 (model answer):')
print('  Fallback action: revert to baseline model and block new entries.')

print('\nNumeric verification:')
print('  metric_name  =', metric_name)
print('  metric_value =', round(float(metric_value), 6))


Interview Question 1 (model answer):
  Formula focus: Logistic Probability
  Formula text : p = 1 / (1 + exp(-z))
  Scenario     : post-earnings drift regime
  Numeric value: 0.512497

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  freeze entries if calibration error worsens across three checks.

Interview Question 3 (model answer):
  I validate timestamps, missing data handling, and assumption consistency before taking risk.

Interview Question 4 (model answer):
  Topic: Model diagnostics and monitoring
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Interview Question 5 (model answer):
  Fallback action: revert to baseline model and block new entries.

Numeric verification:
  metric_name  = up_probability
  metric_value = 0.512497


# Week 06 Day 06: Revision Sprint

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours across recall, rebuild, rerun, and retest blocks.

## 6-10 Hour Daily Structure
- **Core Block 1 (60 min):** Recall formulas, notation, and units without notes.
- **Core Block 2 (75 min):** Rebuild one weekday workflow from memory and compare with reference output.
- **Core Block 3 (60 min):** Rerun at least two notebooks from a clean kernel and capture mismatches.
- **Core Block 4 (60 min):** Diagnose root causes and update safeguards in code/comments.
- **Core Block 5 (45 min):** Interview drill, quiz, and escalation-rule rehearsal.
- **Required Extension Block A (60 min):** Re-test on one alternate ticker universe and one stress window.
- **Required Extension Block B (60 min):** Produce a remediation log with before/after evidence.
- **Optional Deep Work (0-4 hours):** Add robustness tests, monitor drift controls, and harden reporting.

## Why It Matters in Quant
Revision day converts fragile understanding into deployment-ready reliability by forcing recall, rebuild, and guardrail validation.

## Continuity and Handoff
- Previous checkpoint: Week 06 Day 05: Model diagnostics and monitoring
- Previous lesson file: content/week-06/day-05.md
- Today's deliverable: Revision checklist with corrected errors and next-week retest priorities.
- Next handoff target: Week 06 Day 07: Portfolio Project
- Next lesson file: content/week-06/day-07.md

## Revision Plan
- 90 minutes: active recall of weekday concepts and manual formula rewrite.
- 90 minutes: rebuild one weekday code workflow from memory.
- 90 minutes: restart kernel and rerun at least two day notebooks end-to-end.
- 90 minutes: error-log triage, retest plan, and guardrail refinement.
- Optional 0-4 hours: deepen weak areas with extra interview drill and code hardening.

## Focus Areas
- Review tree/forest/boosting differences in one-page summary
- Re-run one ablation experiment from scratch
- Update model-risk checklist

## Mathematical Foundations (LaTeX)
### Formula 1: Forward Target
$$
y_t=\mathbb{1}[r_{t+1}>0]
$$
**Plain-English interpretation:** Label must stay forward-looking.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Use a walk-forward split and verify the metric does not leak future labels into feature construction.

### Formula 2: Logistic Probability
$$
p=\frac{1}{1+e^{-z}}
$$
**Plain-English interpretation:** Convert score to probability.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Use a walk-forward split and verify the metric does not leak future labels into feature construction.

### Formula 3: MSE
$$
\mathcal{L}_{MSE}=\frac{1}{n}\sum_i(y_i-\hat{y}_i)^2
$$
**Plain-English interpretation:** Baseline regression loss.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Use a walk-forward split and verify the metric does not leak future labels into feature construction.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $\hat{y}$ | Model prediction | class/probability | 0.73 |
| $\lambda$ | Regularization strength | non-negative scalar | 0.10 |
| $TP,FP,FN$ | Confusion matrix counts | integer count | TP=52 |

## Real Trading Example
- Instruments: SPY, XLK, XLF, XLE
- Macro overlay (FRED): DFF, VIXCLS
- Suggested window: 2018-01-01 to 2026-03-31
- Stress windows to inspect: 2020-11 to 2021-03, 2022-06 to 2022-11
- Scenario context: post-earnings dispersion with elevated feature drift
- Day objective: Rebuild two weekday workflows and compare decision outputs after a clean rerun.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull listed FRED series and join strictly by release-aware timestamps (no look-ahead).
3. Compute today's formulas and compare calm vs stress-window behavior.
4. Translate outputs into one explicit trade action and one hard risk guardrail.
5. Validate that the decision is consistent with topic 'Revision Sprint'.

## Step-by-Step Solved Problems
### Solved Problem 1: Convert logit to probability
Given:
- Model score z = 1.100.
Solution:
1. $p=\frac{1}{1+e^{-z}}$.
2. p = 1/(1+exp(-1.100)) = 0.750260.
Final answer: Predicted probability = 75.03%.
Common trap: Using forward labels with backward-filled features, which silently introduces leakage.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Compute precision, recall, F1
Given:
- TP=50, FP=12, FN=18.
Solution:
1. $Precision=\frac{TP}{TP+FP}$.
2. Precision=0.8065.
3. $Recall=\frac{TP}{TP+FN}$.
4. Recall=0.7353.
5. $F1=\frac{2PR}{P+R}$.
6. F1=0.7692.
Final answer: Precision=80.65%, Recall=73.53%, F1=76.92%.
Common trap: Using forward labels with backward-filled features, which silently introduces leakage.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Compute ridge objective
Given:
- MSE = 0.0410, ||beta||_2^2 = 1.90, lambda = 0.06.
Solution:
1. $\mathcal{L}=\mathcal{L}_{MSE}+\lambda\|\beta\|_2^2$.
2. L = 0.0410 + 0.06*1.90 = 0.1550.
Final answer: Ridge objective = 0.1550.
Common trap: Using forward labels with backward-filled features, which silently introduces leakage.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Rebuild one workflow from memory, rerun from clean kernel, and document one corrected failure mode.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
X_train, X_valid, y_train, y_valid = temporal_split(features, target)
model = fit_logistic_regression(X_train, y_train, l2=0.1)
proba = model.predict_proba(X_valid)[:, 1]
report = classification_report_from_thresholds(y_valid, proba)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive two key formulas from this week with units and constraints.
2. Rebuild one weekday notebook workflow from memory and verify output parity.
3. Replay one stress regime and document one changed decision rule.
4. Add one extra validation test that would have caught a prior mistake.
5. Record a 60-second explanation of one corrected failure mode.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 06 Day 06): Explain Forward Target and define every symbol clearly for a post-earnings momentum regime where feature drift is elevated.
   - Model answer: "I use Forward Target as a decision bridge from market observations to position sizing. The formula is $y_t=\mathbb{1}[r_{t+1}>0]$. I define each symbol with units first, then compute one concrete value, and finally state what trade action changes because of the result in this regime."
2. Risk manager question: Using one real ticker from this lesson, what hard guardrail would you enforce before live deployment?
   - Model answer: "I would run the workflow on SPY and a stress-sensitive peer, then freeze new model entries when calibration error worsens for 3 consecutive windows. If the guardrail triggers, I switch to paper-trade monitoring and block new risk until diagnostics re-pass."
3. Data integrity question: Which checks must pass before you trust the output and place risk?
   - Model answer: "Before trading I verify strict train/validation timestamp split, no forward-filled target leakage, and feature freshness checks. If any check fails, I classify the run as non-tradable and log the incident."
4. Production question: Why does 'Revision Sprint' matter in live trading systems?
   - Model answer: "Revision Sprint matters because model decisions fail quickly without leakage controls, drift monitoring, and threshold governance. In production I need reproducible calculations, explicit control limits, and escalation rules that survive stress windows."
5. Decision question: If your key metric degrades for three consecutive sessions, what is your fallback plan?
   - Model answer: "I pause new signals, revert to baseline model, and relaunch only after calibration recovers. I only restore risk after rerun evidence confirms that assumptions are stable again."

Scoring rubric:
- 10/10: correct notation, one numeric example, explicit guardrail, data checks, and escalation path.
- 8/10: mostly correct notation plus a clear guardrail and fallback action.
- 6/10: partial correctness but vague controls or missing data validation.
- Below 6/10: formula recall without decision-quality risk controls.

### Interview Drill
- Prompt: "Walk me through Revision Sprint in a model-risk review after prediction drift and weaker precision."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision under constraints.
  3. Awareness of edge cases, costs, and failure modes.
  4. Clear escalation rule when guardrails are breached.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula, assumptions, and validation checks clearly.
  - Decision: explain one actionable rule, one risk guardrail, and one fallback action.

## Required Extension Track (2+ Hours)
- Re-run today's notebook from a fresh kernel so outputs are reproducible without hidden state.
- Add one additional risk guardrail and verify how it changes trade/no-trade decisions.
- Document one failure mode, one mitigation, and one escalation rule for production use.

Extension completion checks:
- [ ] Notebook restarted and all coding cells rerun successfully
- [ ] At least one extra validation/edge-case test added
- [ ] Risk guardrail and fallback action documented

## Reflection Question
Which failure mode still appears after reruns, and what concrete control will prevent it next week?

## Completion Checklist
- [ ] Updated error log entries
- [ ] Weak concepts re-tested
- [ ] Two notebooks rerun from fresh kernel
- [ ] Next-week risk list prepared
- [ ] Interview drill recorded with one fallback action


## Week 06 Day 06 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [17]:
# Revision sprint demo: rebuild weekly core diagnostics
np.random.seed(606)
prices = load_market_prices(['SPY', 'QQQ', 'AAPL'], start='2018-01-01')
returns = prices.pct_change().dropna()

summary = pd.DataFrame({
    'annual_return': returns.mean() * 252,
    'annual_vol': returns.std() * np.sqrt(252),
    'max_drawdown': (prices / prices.cummax() - 1).min(),
})
summary['sharpe_proxy'] = (summary['annual_return'] - 0.03) / summary['annual_vol'].replace(0, np.nan)
summary = summary.sort_values('sharpe_proxy', ascending=False)

print('Revision diagnostics (best risk-adjusted first):')
summary.round(4)


Revision diagnostics (best risk-adjusted first):


,annual_return,annual_vol,max_drawdown,sharpe_proxy
Ticker,,,,
AAPL,0.277000,0.306000,-0.385200,0.807300
QQQ,0.205600,0.238800,-0.351200,0.735400
SPY,0.151700,0.193100,-0.337200,0.630400


## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [18]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(9606)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Revision Sprint',
    'week': 6,
    'day': 6,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Revision Sprint',
 'week': 6,
 'day': 6,
 'observe_annual_return': 0.14923385976701686,
 'observe_annual_vol': 0.20501274117130058,
 'reason_sharpe_proxy': 0.581592437064142,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 06 Day 06 Quiz

Topic: **Revision Sprint**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Explain Logistic Probability (p = 1 / (1 + exp(-z))) and define every symbol clearly.
2. Risk question: What hard guardrail would you enforce before deployment?
3. Data question: Which validation checks must pass before trusting this output?
4. Production question: Why does this topic matter in live systems, not just in notebooks?
5. Escalation question: If the metric degrades for three sessions, what is your fallback action?

Hint for Q2: pause new model entries when calibration error degrades for three windows.
Hint for Q3: feature freshness and no target leakage
Scoring: full credit requires notation correctness, one numeric example, explicit guardrail, and fallback path.

In [19]:
# Quiz model answers (auto-generated)
z = 0.1300
metric_value = 1 / (1 + np.exp(-z))
metric_name = 'up_probability'

print('Interview Question 1 (model answer):')
print('  Formula focus: Logistic Probability')
print('  Formula text : p = 1 / (1 + exp(-z))')
print('  Scenario     : post-earnings drift regime')
print('  Numeric value:', round(float(metric_value), 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  freeze entries if calibration error worsens across three checks.')

print('\nInterview Question 3 (model answer):')
print('  I validate timestamps, missing data handling, and assumption consistency before taking risk.')

print('\nInterview Question 4 (model answer):')
print('  Topic:', 'Revision Sprint')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nInterview Question 5 (model answer):')
print('  Fallback action: revert to baseline model and block new entries.')

print('\nNumeric verification:')
print('  metric_name  =', metric_name)
print('  metric_value =', round(float(metric_value), 6))


Interview Question 1 (model answer):
  Formula focus: Logistic Probability
  Formula text : p = 1 / (1 + exp(-z))
  Scenario     : post-earnings drift regime
  Numeric value: 0.532454

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  freeze entries if calibration error worsens across three checks.

Interview Question 3 (model answer):
  I validate timestamps, missing data handling, and assumption consistency before taking risk.

Interview Question 4 (model answer):
  Topic: Revision Sprint
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Interview Question 5 (model answer):
  Fallback action: revert to baseline model and block new entries.

Numeric verification:
  metric_name  = up_probability
  metric_value = 0.532454


# Week 06 Day 07: Portfolio Project

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours for implementation, validation, and communication drills.

## 6-10 Hour Daily Structure
- **Core Block 1 (60 min):** Restate objective, assumptions, and measurable success criteria.
- **Core Block 2 (75 min):** Build/clean data pipeline and verify timestamp integrity.
- **Core Block 3 (75 min):** Implement project logic and validate formulas against spot checks.
- **Core Block 4 (60 min):** Produce diagnostics, stress checks, and fallback pathways.
- **Core Block 5 (45 min):** Deliver interview-style defense with risk controls and escalation.
- **Required Extension Block A (60 min):** Re-run project on alternate assumptions and compare drift.
- **Required Extension Block B (60 min):** Prepare production memo and launch/no-launch decision log.
- **Optional Deep Work (0-4 hours):** Expand tests, improve monitoring, and polish stakeholder narrative.

## Why It Matters in Quant
Project day is where research quality meets execution discipline and communication quality under risk constraints.

## Continuity and Handoff
- Previous checkpoint: Week 06 Day 06: Revision Sprint
- Previous lesson file: content/week-06/day-06.md
- Today's deliverable: Ensemble model comparison
- Next handoff target: Week 07 Day 01: Unsupervised problem framing
- Next lesson file: content/week-07/day-01.md

## Project Blueprint
### Project Title
Ensemble model comparison

### Problem Statement
Compare tree-based models under a consistent validation and monitoring framework.

### Data Sources
- Price features
- Volume features
- Regime labels

### Implementation Steps
1. Create baseline feature set
2. Train tree/forest/boosting models
3. Run ablation and stability tests
4. Evaluate by regime
5. Document deployment recommendation

### Evaluation Metrics
- Validation score
- Stability score
- Calibration error
- Drift sensitivity

### Execution Standard
- [ ] Notebook/script runs from clean start without hidden state
- [ ] Outputs include at least one diagnostic table and one chart
- [ ] One explicit risk guardrail and fallback action are documented

### Deliverables
- Notebook or script output
- One-page summary memo
- Tracker update with completion and lessons learned

## Mathematical Foundations (LaTeX)
### Formula 1: Logistic Probability
$$
p=\frac{1}{1+e^{-z}}
$$
**Plain-English interpretation:** Convert score to probability.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Use a walk-forward split and verify the metric does not leak future labels into feature construction.

### Formula 2: MSE
$$
\mathcal{L}_{MSE}=\frac{1}{n}\sum_i(y_i-\hat{y}_i)^2
$$
**Plain-English interpretation:** Baseline regression loss.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Use a walk-forward split and verify the metric does not leak future labels into feature construction.

### Formula 3: Cross-Entropy
$$
\mathcal{L}_{CE}=-\frac{1}{n}\sum_i[y_i\log p_i+(1-y_i)\log(1-p_i)]
$$
**Plain-English interpretation:** Classification objective.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Use a walk-forward split and verify the metric does not leak future labels into feature construction.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $\hat{y}$ | Model prediction | class/probability | 0.73 |
| $\lambda$ | Regularization strength | non-negative scalar | 0.10 |
| $TP,FP,FN$ | Confusion matrix counts | integer count | TP=52 |

## Real Trading Example
- Instruments: SPY, XLK, XLF, XLE
- Macro overlay (FRED): DFF, VIXCLS
- Suggested window: 2018-01-01 to 2026-03-31
- Stress windows to inspect: 2020-11 to 2021-03, 2022-06 to 2022-11
- Scenario context: post-earnings dispersion with elevated feature drift
- Day objective: Deliver a capstone-quality notebook and summarize one trade-off under stress assumptions.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull listed FRED series and join strictly by release-aware timestamps (no look-ahead).
3. Compute today's formulas and compare calm vs stress-window behavior.
4. Translate outputs into one explicit trade action and one hard risk guardrail.
5. Validate that the decision is consistent with topic 'Portfolio Project'.

## Step-by-Step Solved Problems
### Solved Problem 1: Convert logit to probability
Given:
- Model score z = 1.400.
Solution:
1. $p=\frac{1}{1+e^{-z}}$.
2. p = 1/(1+exp(-1.400)) = 0.802184.
Final answer: Predicted probability = 80.22%.
Common trap: Using forward labels with backward-filled features, which silently introduces leakage.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Compute precision, recall, F1
Given:
- TP=50, FP=12, FN=18.
Solution:
1. $Precision=\frac{TP}{TP+FP}$.
2. Precision=0.8065.
3. $Recall=\frac{TP}{TP+FN}$.
4. Recall=0.7353.
5. $F1=\frac{2PR}{P+R}$.
6. F1=0.7692.
Final answer: Precision=80.65%, Recall=73.53%, F1=76.92%.
Common trap: Using forward labels with backward-filled features, which silently introduces leakage.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Compute ridge objective
Given:
- MSE = 0.0410, ||beta||_2^2 = 1.90, lambda = 0.06.
Solution:
1. $\mathcal{L}=\mathcal{L}_{MSE}+\lambda\|\beta\|_2^2$.
2. L = 0.0410 + 0.06*1.90 = 0.1550.
Final answer: Ridge objective = 0.1550.
Common trap: Using forward labels with backward-filled features, which silently introduces leakage.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Ship the project notebook with reproducible outputs, controls, and one escalation rule.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
X_train, X_valid, y_train, y_valid = temporal_split(features, target)
model = fit_logistic_regression(X_train, y_train, l2=0.1)
proba = model.predict_proba(X_valid)[:, 1]
report = classification_report_from_thresholds(y_valid, proba)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. State project objective and one hard failure condition in under 45 seconds.
2. Validate one formula output against a manual spot-check.
3. Show one stress scenario where your decision changes and explain why.
4. Add one edge-case test and one fallback rule to the notebook.
5. Deliver a one-minute PM summary with one risk guardrail.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 06 Day 07): Explain Logistic Probability and define every symbol clearly for a post-earnings momentum regime where feature drift is elevated.
   - Model answer: "I use Logistic Probability as a decision bridge from market observations to position sizing. The formula is $p=\frac{1}{1+e^{-z}}$. I define each symbol with units first, then compute one concrete value, and finally state what trade action changes because of the result in this regime."
2. Risk manager question: Using one real ticker from this lesson, what hard guardrail would you enforce before live deployment?
   - Model answer: "I would run the workflow on SPY and a stress-sensitive peer, then freeze new model entries when calibration error worsens for 3 consecutive windows. If the guardrail triggers, I switch to paper-trade monitoring and block new risk until diagnostics re-pass."
3. Data integrity question: Which checks must pass before you trust the output and place risk?
   - Model answer: "Before trading I verify strict train/validation timestamp split, no forward-filled target leakage, and feature freshness checks. If any check fails, I classify the run as non-tradable and log the incident."
4. Production question: Why does 'Portfolio Project' matter in live trading systems?
   - Model answer: "Portfolio Project matters because model decisions fail quickly without leakage controls, drift monitoring, and threshold governance. In production I need reproducible calculations, explicit control limits, and escalation rules that survive stress windows."
5. Decision question: If your key metric degrades for three consecutive sessions, what is your fallback plan?
   - Model answer: "I pause new signals, revert to baseline model, and relaunch only after calibration recovers. I only restore risk after rerun evidence confirms that assumptions are stable again."

Scoring rubric:
- 10/10: correct notation, one numeric example, explicit guardrail, data checks, and escalation path.
- 8/10: mostly correct notation plus a clear guardrail and fallback action.
- 6/10: partial correctness but vague controls or missing data validation.
- Below 6/10: formula recall without decision-quality risk controls.

### Interview Drill
- Prompt: "Walk me through Portfolio Project in a model-risk review after prediction drift and weaker precision."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision under constraints.
  3. Awareness of edge cases, costs, and failure modes.
  4. Clear escalation rule when guardrails are breached.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula, assumptions, and validation checks clearly.
  - Decision: explain one actionable rule, one risk guardrail, and one fallback action.

## Required Extension Track (2+ Hours)
- Re-run today's notebook from a fresh kernel so outputs are reproducible without hidden state.
- Add one additional risk guardrail and verify how it changes trade/no-trade decisions.
- Document one failure mode, one mitigation, and one escalation rule for production use.

Extension completion checks:
- [ ] Notebook restarted and all coding cells rerun successfully
- [ ] At least one extra validation/edge-case test added
- [ ] Risk guardrail and fallback action documented

## Reflection Question
What single risk control would block launch today, and what evidence would clear it?

## Completion Checklist
- [ ] Project notebook runs cleanly from fresh kernel
- [ ] Risk guardrail and fallback action documented
- [ ] Stress scenario comparison completed
- [ ] One-page summary memo finalized
- [ ] Launch/no-launch decision recorded with evidence


## Week 06 Day 07 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [20]:
# Project day demo: mini portfolio report with trade recommendation
np.random.seed(607)
assets = ['SPY', 'QQQ', 'TLT', 'GLD']
prices = load_market_prices(assets, start='2019-01-01')
returns = prices.pct_change().dropna()

annual_return = returns.mean() * 252
annual_vol = returns.std() * np.sqrt(252)
score = (annual_return - 0.03) / annual_vol.replace(0, np.nan)

report = pd.DataFrame({
    'annual_return': annual_return,
    'annual_vol': annual_vol,
    'sharpe_proxy': score,
}).sort_values('sharpe_proxy', ascending=False)

top_asset = report.index[0]
print('Project summary:')
print(report.round(4))
print(f"\nSuggested focus asset for follow-up research: {top_asset}")


Project summary:
        annual_return  annual_vol  sharpe_proxy
Ticker                                         
GLD          0.194100    0.172700      0.950000
QQQ          0.232200    0.240200      0.841900
SPY          0.177800    0.196000      0.754000
TLT         -0.005100    0.162600     -0.215600

Suggested focus asset for follow-up research: GLD


## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [21]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(9607)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Portfolio Project',
    'week': 6,
    'day': 7,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Portfolio Project',
 'week': 6,
 'day': 7,
 'observe_annual_return': 0.14923385976701686,
 'observe_annual_vol': 0.20501274117130058,
 'reason_sharpe_proxy': 0.581592437064142,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 06 Day 07 Quiz

Topic: **Portfolio Project**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Explain Logistic Probability (p = 1 / (1 + exp(-z))) and define every symbol clearly.
2. Risk question: What hard guardrail would you enforce before deployment?
3. Data question: Which validation checks must pass before trusting this output?
4. Production question: Why does this topic matter in live systems, not just in notebooks?
5. Escalation question: If the metric degrades for three sessions, what is your fallback action?

Hint for Q2: pause new model entries when calibration error degrades for three windows.
Hint for Q3: feature freshness and no target leakage
Scoring: full credit requires notation correctness, one numeric example, explicit guardrail, and fallback path.

In [22]:
# Quiz model answers (auto-generated)
z = 0.2100
metric_value = 1 / (1 + np.exp(-z))
metric_name = 'up_probability'

print('Interview Question 1 (model answer):')
print('  Formula focus: Logistic Probability')
print('  Formula text : p = 1 / (1 + exp(-z))')
print('  Scenario     : post-earnings drift regime')
print('  Numeric value:', round(float(metric_value), 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  freeze entries if calibration error worsens across three checks.')

print('\nInterview Question 3 (model answer):')
print('  I validate timestamps, missing data handling, and assumption consistency before taking risk.')

print('\nInterview Question 4 (model answer):')
print('  Topic:', 'Portfolio Project')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nInterview Question 5 (model answer):')
print('  Fallback action: revert to baseline model and block new entries.')

print('\nNumeric verification:')
print('  metric_name  =', metric_name)
print('  metric_value =', round(float(metric_value), 6))


Interview Question 1 (model answer):
  Formula focus: Logistic Probability
  Formula text : p = 1 / (1 + exp(-z))
  Scenario     : post-earnings drift regime
  Numeric value: 0.552308

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  freeze entries if calibration error worsens across three checks.

Interview Question 3 (model answer):
  I validate timestamps, missing data handling, and assumption consistency before taking risk.

Interview Question 4 (model answer):
  Topic: Portfolio Project
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Interview Question 5 (model answer):
  Fallback action: revert to baseline model and block new entries.

Numeric verification:
  metric_name  = up_probability
  metric_value = 0.552308
